In [1]:
import os
import asyncio
import logging
from typing import Optional, Dict, Any

from dotenv import load_dotenv
from google import genai
from google.genai import types


In [2]:
print("cwd:", os.getcwd())
print(".env exists:", os.path.exists(".env"))

cwd: c:\Users\Nolan\Documents\My dokuments\GoIT\AI Agents\ai_agents_for_beginers
.env exists: True


In [3]:
load_dotenv() # Load environment variables from .env file

api_key = os.getenv("GEMINI_API_KEY")
print('API Key loaded successfully:', bool(api_key))

client = genai.Client(api_key=api_key)

API Key loaded successfully: True


In [4]:
for m in client.models.list():
    print(m.name)


models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-2.5-flash-lite-preview-09-2025
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-robotics-er-1.5-preview
models/gemini-2.5-computer-use-preview-10-2025
models/deep-research-pro-preview-12-2025
models/gemini-embedding-001
models/aqa
models/imagen-4.0-generate-001
models/imagen-4.0-ultra-generate-001
models/imagen-4.0-fast

In [5]:
response = client.models.generate_content(
    model = "gemini-3-flash-preview",
    contents='Скажи одним реченням: що таке AI-агент?'
)

print(response.text)

**ШІ-агент** — це автономна інтелектуальна система, яка здатна самостійно сприймати середовище, приймати рішення та виконувати послідовність дій для досягнення поставленої мети без постійного втручання людини.


In [12]:
def suggest_destinations(budget: int, nights: int, month: str, kids_ages: list[int]) -> str:
    print(f"[TOOL CALL] budget={budget}, nights={nights}, month={month}, kids_ages={kids_ages}")
    """
    Tool: рекомендації напрямків для сімейної поїздки.
    total_budget_usd — загальний бюджет на всю поїздку (переліт+житло+частково витрати).
    nights — кількість ночей.
    month — місяць (наприклад "липень").
    kids_ages — список віків дітей, напр. [4, 1].
    """
    kids_count = len(kids_ages)
    yangest = max(kids_ages) if kids_ages else None
    budget_per_night = budget / nights
    if budget_per_night < 100:
        tier = "low"
    elif budget_per_night < 150:
        tier = "mid"
    else:
        tier = "high"

    is_hot_month = month.lower() in ["червень", "липень", "серпень"]
    has_toddlers = (yangest is not None and yangest < 3)

    option = []
    if has_toddlers:
        if tier == "low":
            options = [
                "Польща: Балтика (коротка логістика, апартаменти)",
                "Словаччина: Татри + терми (спокійно, без довгих перельотів)",
                "Чехія: Прага + дитячі активності (місто, але комфортно)"
            ]
        elif tier == "mid":
            options = [
                "Італія: Ріміні/Ліньяно (сімейні курорти, зручний захід у море)",
                "Хорватія: Істрія (море + короткі переїзди)",
                "Греція: Крит (сімейні готелі, пляжі)"
            ]
        else:  # high
            options = [
                "Іспанія: Коста-Дорада (сімейні готелі + парки)",
                "Греція: Крит/Родос (вищий комфорт, all inclusive)",
                "Італія: Сардинія (якісні пляжі, але дорожче)"
            ]
    else:
        if tier == "low":
            options = [
                "Угорщина: Будапешт + терми + зоопарк",
                "Австрія/Словаччина: озера/гори (активний відпочинок)",
                "Хорватія: бюджетні апартаменти ближче до моря"
            ]
        elif tier == "mid":
            options = [
                "Іспанія: Барселона + море",
                "Португалія: Лісабон + океан",
                "Італія: Рим + узбережжя"
            ]
        else:
            options = [
                "Португалія: Мадейра (комфорт + природа)",
                "Іспанія: Канари (стабільна погода, сімейний формат)",
                "Італія: Тоскана + море (комфорт, авто-логістика)"
            ]

    note = []
    note.append(f"Бюджет/ніч ≈ ${budget_per_night:.0f} ({tier}).")
    if is_hot_month:
        note.append("Липень/серпень: перевір кондиціонер у житлі та уникай довгих переїздів у спеку.")
    if has_toddlers:
        note.append("З малими дітьми краще коротший переліт і апартаменти з кухнею/пральною.")

    return (
        f"Діти: {kids_count} (вік: {kids_ages}). Ночей: {nights}. Бюджет: ${budget}.\n"
        f"Місяць: {month}.\n\n"
        "Рекомендації:\n- " + "\n- ".join(options) +
        "\n\nНотатки:\n- " + "\n- ".join(note)
    )


In [7]:
print(suggest_destinations(1500, 10, "липень", [5, 3]))

Діти: 2 (вік: [5, 3]). Ночей: 10. Бюджет: $1500.
Місяць: липень.

Рекомендації:
- Португалія: Мадейра (комфорт + природа)
- Іспанія: Канари (стабільна погода, сімейний формат)
- Італія: Тоскана + море (комфорт, авто-логістика)

Нотатки:
- Бюджет/ніч ≈ $150 (high).
- Липень/серпень: перевір кондиціонер у житлі та уникай довгих переїздів у спеку.


In [26]:
prompt = (
    "Порадь 3 напрямки для сімейної відпустки. "
    "Параметри: загальна сума 1000,  тривалість 8 днів , в липні, з двома дітьми [5,3]. "
    "Використай інструмент suggest_destinations."
)
# Вимикаємо automatic function calling, щоб побачити function_call
config = types.GenerateContentConfig(
    tools=[suggest_destinations],  # Python-функція як tool
    automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=False),
)

response = client.models.generate_content(
    model="models/gemini-3-flash-preview",
    contents=prompt,
    config=config
)

print(response.text)
# print("\n--- STREAM ---\n")
# stream = client.models.generate_content_stream(
#     model="models/gemini-2.5-flash",
#     contents=prompt,
#     config=types.GenerateContentConfig(tools=[suggest_destinations]),
# )

# for chunk in stream: 
#     if getattr(chunk, "text", None): 
#         print(chunk.text, end="", flush=True)

# print("\n\n--- END ---")


[TOOL CALL] budget=1000, nights=8, month=July, kids_ages=[5, 3]
Ось 3 напрямки для вашої сімейної відпустки в липні, підібрані з урахуванням бюджету та віку дітей (5 і 3 роки):

1. **Іспанія: Барселона та узбережжя (Коста-Дорада)**
   * **Чому підходить:** Барселона пропонує чудові парки (наприклад, парк Цитаделі), океанаріум та інтерактивний музей науки CosmoCaixa, який обожнюють діти. Поруч знаходяться містечка з широкими піщаними пляжами та мілким входом у воду (Салоу або Калафель), що ідеально для малечі 3-5 років.
   * **Бюджет:** Оскільки бюджет близько $125 на добу на всю родину, варто розглянути апартаменти з власною кухнею та вибирати житло трохи далі від самого центру Барселони.

2. **Португалія: Лісабон та Кашкайш**
   * **Чому підходить:** Лісабонський океанаріум — один з найкращих у світі. Поруч розташоване містечко Кашкайш з тихими пляжами без великих хвиль. Поїздка на старовинному жовтому трамваї стане справжньою пригодою для дітей.
   * **Бюджет:** Липень — пік сезону, 